# this is for constructing the AI prompt. Do NOT run all without understanding the code below or talking with Noah

In [1]:
#Set up
from pathlib import Path
import json
import os
import sys
import time
import warnings
import numpy as np
import logging


import pandas as pd
from IPython.display import display

rng = np.random.default_rng(seed=1312)
random_texts = rng.integers(1, 1061, size=35)

test_texts = pd.read_csv("full_text.csv", dtype={"id": str}).iloc[random_texts]
print(len(test_texts), "texts")  # Check: 50 texts
test_texts["president"].value_counts()

35 texts


president
Andrew Jackson           4
James K. Polk            3
Rutherford B. Hayes      3
Lyndon B. Johnson        3
Ronald Reagan            2
Andrew Johnson           2
Richard M. Nixon         2
Abraham Lincoln          2
Chester A. Arthur        1
Bill Clinton             1
Thomas Jefferson         1
George W. Bush           1
John F. Kennedy          1
Zachary Taylor           1
Benjamin Harrison        1
Donald Trump             1
Grover Cleveland         1
James Monroe             1
Franklin D. Roosevelt    1
Woodrow Wilson           1
Theodore Roosevelt       1
Barack Obama             1
Name: count, dtype: int64

In [2]:
warnings.filterwarnings("ignore", category=FutureWarning)  # a harmless warning from one of the libraries

pd.set_option("display.max_colwidth", 300)  # show more of each text

SUPPORTED_PYTHON = (3, 13)
if sys.version_info[:2] != SUPPORTED_PYTHON:
    print(
        "Setup note: this course is tested with Python 3.13; "
        f"you are running {sys.version.split()[0]}."
    )

def find_course_root():
    """Find the cloned repository when this notebook is running locally."""
    for folder in (Path.cwd(), *Path.cwd().parents):
        if (folder / "pyproject.toml").exists() and (folder / "data").is_dir():
            return folder
    return None

LOCAL_COURSE_ROOT = find_course_root()
COURSE_ROOT = LOCAL_COURSE_ROOT or Path.cwd()
DATA_DIR = (
    LOCAL_COURSE_ROOT / "data"
    if LOCAL_COURSE_ROOT
    else COURSE_ROOT / "compss211_data"
)
GENERATED_DIR = COURSE_ROOT / "generated"
DATA_DIR.mkdir(parents=True, exist_ok=True)
GENERATED_DIR.mkdir(parents=True, exist_ok=True)

DATA_BASE_URL = (
    "https://raw.githubusercontent.com/"
    "macss-berkeley/compss-211a/main/data"
)

def course_data_path(filename):
    """Use a local course file, or download it when running in Colab."""
    path = DATA_DIR / filename
    if not path.exists():
        from urllib.request import urlretrieve

        urlretrieve(f"{DATA_BASE_URL}/{filename}", path)
    return path

print(f"Python {sys.version.split()[0]} | data={DATA_DIR}")

Setup note: this course is tested with Python 3.13; you are running 3.14.7.
Python 3.14.7 | data=c:\Users\cgsta\Documents\Berkeley\Computing I\Pres speech\Miller-211A\ai_junk\compss211_data


In [3]:
print(COURSE_ROOT)

c:\Users\cgsta\Documents\Berkeley\Computing I\Pres speech\Miller-211A\ai_junk


In [4]:
test_texts['id'] = test_texts.index.astype(str)

In [5]:
test_texts['label'] = 'ignore'

### Ai set-up

In [6]:
# Supplied: connect to Gemini. The key is read from your .env file and never printed.
import logging

from dotenv import load_dotenv
from google import genai
from google.genai import errors, types

load_dotenv(COURSE_ROOT / ".env")
key = os.environ.get("GEMINI_API_KEY", "")
if not key or key.startswith("paste-"):
    raise ValueError("Add GEMINI_API_KEY to your .env file, then run this cell again.")

client = genai.Client(api_key=key)
GEMINI_MODEL = "gemini-3.5-flash-lite"  # use the replacement model if bCourses announces one

# Hide a warning that the google-genai package prints on every call. It does not mean anything went wrong.
logging.getLogger("google_genai.types").setLevel(logging.ERROR)
print("Connected to Gemini.")

Connected to Gemini.


In [7]:
RETRY_CODES = {429, 503}


def ask_gemini(prompt, system, schema, max_tries=4):
    """Send one prompt with this system instruction and schema. Return the label and the reason. Try again after a temporary error."""
    wait = 10  # seconds to wait before the first retry
    for attempt in range(1, max_tries + 1):
        try:
            response = client.models.generate_content(
                model=GEMINI_MODEL,
                contents=prompt,
                config=types.GenerateContentConfig(
                    system_instruction=system,
                    response_mime_type="application/json",
                    response_schema=schema,
                ),
            )
            time.sleep(6)  # wait between calls to stay under the free tier's limit per minute
            return json.loads(response.text)
        except errors.APIError as error:
            if error.code not in RETRY_CODES or attempt == max_tries:
                raise  # stop and show the error
            print(f"Error {error.code}. Waiting {wait} seconds, then trying again.")
            time.sleep(wait)
            wait = wait * 2  # wait longer each time: 10, 20, 40 seconds


def label_texts(texts, system, schema, path):
    """Ask Gemini about every text in texts. Save each answer to path right away."""
    # Texts that are already in the file were labeled in an earlier run, so skip them.
    done = set(pd.read_csv(path, dtype={"id": str})["id"]) if path.exists() else set()
    for text in texts.itertuples():
        if text.id in done:
            continue
        reply = ask_gemini(f"Text: {text.text}", system, schema)
        row = pd.DataFrame([{"id": text.id, "label": text.label, **reply, "model": GEMINI_MODEL}])
        # mode="a" adds one line to the end of the file. Write the column names only if the file is new.
        row.to_csv(path, mode="a", header=not path.exists(), index=False)
        done.add(text.id)
        print(len(done), "of", len(texts), "done")
    results = pd.read_csv(path, dtype={"id": str, "label": str, "llm_label": str})
    return results[results["id"].isin(texts["id"])]

In [8]:
PROJECT_SYSTEM = """You label texts from preidental speeches. One text is one row within the [text].

Give the text one label:
- welcoming: the text expresses a positive attitude towards immigration and immigrants/refugees, emphasizes American openness towards migrants, or calls for humanitarian support for migrants/refugees.
- restrictive: the text expresses a negative attitude towards immigration and immigrants/refugees; emphasizes border security, legal limits, deportation, or national security concerns regarding immigration; or calls for stricter immigration policies.
- neutral: the text does not express a clear positive or negative attitude towards immigration and immigrants/refugees; it is neither particularly welcoming nor restrictive.
- unclear: the text does not contain enough context to determine the attitude towards immigration and immigrants/refugees.
- ⁠not relevant: the text is functionally not related to immigration or immigrants/refugees.

Also give a reason of one short sentence.""" 
PROJECT_LABELS = ["welcoming", "restrictive",'neutral', "unclear", 'not relevant']

assert "..." not in PROJECT_SYSTEM and "..." not in PROJECT_LABELS, "Fill in PROJECT_SYSTEM and PROJECT_LABELS."
PROJECT_SCHEMA = {
    "type": "object",
    "properties": {
        "llm_label": {"type": "string", "enum": [str(label) for label in PROJECT_LABELS]},  # labels must be text, so 1 becomes "1"
        "llm_reason": {"type": "string"},
    },
    "required": ["llm_label", "llm_reason"],
}

In [9]:
write_texts = test_texts
score_texts = pd.read_csv('full_text.csv')
print(len(write_texts), "texts to write the prompt on;", len(score_texts), "to score it on")  # Check: 20 and 30

35 texts to write the prompt on; 1059 to score it on


In [10]:
first_results = label_texts(write_texts, PROJECT_SYSTEM, PROJECT_SCHEMA, GENERATED_DIR / "full_text.csv")
merged_results = write_texts.merge(first_results[['id', 'llm_label', 'llm_reason']], on='id')
merged_results[['president', 'year', 'title', 'speech_type', 'llm_label', 'llm_reason']].head()

,president,year,title,speech_type,llm_label,llm_reason
0,Andrew Jackson,1831,"February 22, 1831: Message Regarding Indian Relations",Message,not relavent,"The text discusses federal jurisdiction, state laws, and historical relations regarding Native American tribes, which falls under Indian affairs rather than immigration."
1,Andrew Jackson,1831,"February 22, 1831: Message Regarding Indian Relations",Message,not relavent,"The text discusses federal jurisdiction, state sovereignty, and Indian affairs, which is legally distinct from modern international immigration."
2,Andrew Jackson,1831,"February 22, 1831: Message Regarding Indian Relations",Message,not relavent,"The text discusses the jurisdictional rights of states over Native American tribes and Indian affairs, which is legally distinct from modern international immigration."
3,Chester A. Arthur,1884,"December 10, 1884: Message Regarding Central American Canal",Message,not relavent,The text discusses the construction of an interoceanic canal in Nicaragua and is not about immigration or immigrants.
4,Chester A. Arthur,1884,"December 10, 1884: Message Regarding Central American Canal",Message,not relavent,The text discusses the construction of an interoceanic canal in Nicaragua rather than immigration.


In [11]:
merged_results['llm_label'].value_counts()

llm_label
not relavent    33
welcoming        6
restrictive      2
neutral          1
Name: count, dtype: int64

In [12]:
merged_results.to_csv("test_text.csv", index=False)